# Unwrapping AI
## Zero to Transformer — Step 03: Linear Regression

**Creator:** Nitin K. Sethi  
**Learning milestone:** 03 of 12  
**Environment:** Google Colab / Python

> Learn the concept. Build the model. Understand the why.

### Today's objective
Build a regression model that predicts California house values from housing features. Start with one input feature, then use all eight features.

### What you will learn
1. The linear regression equation: `y_hat = b0 + b1*x`.
2. How slope and intercept describe a fitted line.
3. How to fit simple and multiple linear regression with scikit-learn.
4. How to compare a model with a simple mean-prediction baseline.
5. How to interpret predictions, errors, and coefficients.

**Important:** This is a learning experiment, not a real-estate valuation tool. The dataset is historical and its target values are capped.

## 1. Concept: what is linear regression?

Linear regression estimates a relationship between one or more input features and a continuous target.

For one feature:

\[
\hat{y} = b_0 + b_1x
\]

- `b0` is the intercept: the predicted target when `x = 0`.
- `b1` is the slope: the estimated change in the prediction for a one-unit increase in `x`.

For multiple features:

\[
\hat{y} = b_0 + b_1x_1 + b_2x_2 + \dots + b_px_p
\]

The model chooses coefficients that minimize the sum of squared errors on the training data (ordinary least squares).

**Remember:** a fitted relationship does not prove causation. Coefficients are also affected by feature units and correlations between features.

## 2. Import libraries and record versions

We use NumPy and pandas for data handling, Matplotlib for visualizations, and scikit-learn for the dataset, splitting, models, and evaluation.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## 3. Load the California Housing dataset

The dataset has 20,640 observations and eight numeric input features. The target is median house value, expressed in units of **$100,000**.

The dataset is downloaded by scikit-learn the first time it is requested. Google Colab needs internet access for this download.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

print("Dataset shape:", df.shape)
print("Feature count:", X.shape[1])
print("Target name:", housing.target_names[0])
display(df.head())

### Inspect the data

Check missing values and review basic statistics before fitting a model.

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
display(df.describe().T)

## 4. Split the data

We reserve 20% of the observations for testing. The fixed `random_state` makes the split reproducible.

The test set must stay separate from model fitting. We will use it only to evaluate how well the trained models perform on unseen observations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training proportion:", round(len(X_train) / len(X), 2))
print("Testing proportion:", round(len(X_test) / len(X), 2))

assert len(X) == 20640
assert len(X_train) == 16512
assert len(X_test) == 4128
assert X_train.index.intersection(X_test.index).empty

## 5. Build a simple linear regression model

First, use only `MedInc` (median income) to predict the target. This lets us see a two-dimensional relationship and understand a fitted line.

The coefficient is an association estimated by the model; it should not be interpreted as proof that changing income causes a particular house-value change.

In [ ]:
simple_feature = ["MedInc"]

simple_model = LinearRegression()
simple_model.fit(X_train[simple_feature], y_train)

simple_predictions = simple_model.predict(X_test[simple_feature])

print("Intercept:", simple_model.intercept_)
print("Slope for MedInc:", simple_model.coef_[0])
print("Simple model test MSE:", mean_squared_error(y_test, simple_predictions))
print("Simple model test R²:", r2_score(y_test, simple_predictions))

### Visualize the fitted line

The scatter points below are test observations. The line shows the model's prediction across the observed test-set income range.

In [ ]:
x_line = np.linspace(
    X_test["MedInc"].min(),
    X_test["MedInc"].max(),
    200
)
y_line = simple_model.predict(pd.DataFrame({"MedInc": x_line}))

plt.figure(figsize=(9, 5))
plt.scatter(X_test["MedInc"], y_test, alpha=0.18, s=12, label="Actual test observations")
plt.plot(x_line, y_line, linewidth=2.5, label="Fitted linear regression")
plt.xlabel("Median income (dataset units)")
plt.ylabel("Median house value (units of $100,000)")
plt.title("Unwrapping AI — Simple Linear Regression")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Build a multiple linear regression model

Now use all eight input features. This is still linear regression: the model estimates one coefficient per feature, plus an intercept.

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)

multiple_predictions = multiple_model.predict(X_test)

print("Intercept:", multiple_model.intercept_)
print("Number of coefficients:", len(multiple_model.coef_))

### Inspect the coefficients

Each coefficient is the model's estimated change in predicted target for a one-unit increase in that feature, **holding the other included features constant**.

The features use different units and scales, so comparing raw coefficient magnitudes to rank feature importance is not valid.

In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": multiple_model.coef_
}).sort_values("Coefficient", key=np.abs, ascending=False)

display(coefficient_table)

## 7. Evaluate against a baseline

A useful model should be compared with a simple reference. Our baseline predicts the **training-set mean** for every test observation.

We report:
- **MSE:** average squared error (target units squared).
- **RMSE:** square root of MSE (same units as the target).
- **MAE:** average absolute error (same units as the target).
- **R²:** proportion of target variance explained relative to predicting the test-set mean; it can be negative.

Lower MSE, RMSE, and MAE are better. Higher R² is generally better.

In [ ]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

def regression_metrics(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "Model": model_name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred)
    }

results = pd.DataFrame([
    regression_metrics(y_test, baseline_predictions, "Mean baseline"),
    regression_metrics(y_test, simple_predictions, "Simple: MedInc"),
    regression_metrics(y_test, multiple_predictions, "Multiple: 8 features")
])

display(results.round(4))

### Verification: compare the models

The multiple-feature model will often outperform the one-feature model, but do not assume that in advance. Use the actual results from this run.

The key baseline check is whether the fitted model improves on predicting the training-set mean. A model can have a negative test R² if it performs worse than the test-set mean reference.

In [ ]:
assert np.isfinite(results[["MSE", "RMSE", "MAE", "R²"]].to_numpy()).all()
assert np.allclose(
    results.loc[0, "RMSE"] ** 2,
    results.loc[0, "MSE"],
    rtol=1e-8,
    atol=1e-10
)

print("Metric calculations are finite.")
print("RMSE squared matches MSE for the baseline.")
print("Compare the test metrics above; do not claim an improvement unless the values show it.")

## 8. Visualize multiple-model predictions and residuals

A residual is `actual - predicted`. Residual plots help reveal patterns that a single summary metric can hide.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, multiple_predictions, alpha=0.2, s=12)
low = min(y_test.min(), multiple_predictions.min())
high = max(y_test.max(), multiple_predictions.max())
plt.plot([low, high], [low, high], linestyle="--", linewidth=2)
plt.xlabel("Actual house value (units of $100,000)")
plt.ylabel("Predicted house value (units of $100,000)")
plt.title("Actual vs Predicted — Multiple Linear Regression")
plt.grid(alpha=0.25)
plt.show()

residuals = y_test - multiple_predictions

plt.figure(figsize=(9, 5))
plt.scatter(multiple_predictions, residuals, alpha=0.2, s=12)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("Predicted house value (units of $100,000)")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residuals vs Predicted Values")
plt.grid(alpha=0.25)
plt.show()

## 9. Interpret the experiment

Complete these observations after reviewing your own output:

1. Did the simple `MedInc` model outperform the mean baseline on test MSE and test R²?
2. Did the multiple-feature model improve on the simple model in this run?
3. Do the residuals appear randomly scattered around zero, or is there a visible pattern?
4. Why should we avoid interpreting a coefficient as proof of causation?
5. What does the target unit ($100,000) imply for the interpretation of RMSE?

**Do not copy generic claims. Write observations that match the metrics and plots generated in your run.**

## 10. Final verification checklist

Before marking Step 03 complete, confirm all of the following:

- [ ] Dataset loaded: 20,640 rows and 8 features.
- [ ] Train/test split: 16,512 training rows and 4,128 test rows.
- [ ] Simple linear regression fitted using `MedInc`.
- [ ] Multiple linear regression fitted using all 8 features.
- [ ] Mean baseline, MSE, RMSE, MAE, and R² calculated.
- [ ] Both prediction visualizations and residual plot displayed.
- [ ] No cell errors; all metric values are finite.
- [ ] I wrote observations based on my actual output.
- [ ] Notebook saved with outputs after successful Colab execution.

### GitHub closeout

Save the executed notebook to the existing `zero-to-transformer` repository using this filename:

`Unwrapping AI — Zero to Transformer — Step 03 — Linear Regression.ipynb`

Suggested commit message:

`Add Step 03: Linear Regression | Unwrapping AI`

Then update the README roadmap only after the notebook has been executed and committed:

- Step 01 — Train–Test Split: Complete
- Step 02 — Mean Squared Error: Complete
- Step 03 — Linear Regression: Complete (only after verification and GitHub confirmation)
- Step 04 — Gradient Descent: Planned / Next

Suggested README commit message:

`Update roadmap: Complete Step 03 — Linear Regression`

### Dependencies

This notebook uses:
- numpy
- pandas
- matplotlib
- scikit-learn

Record the versions printed in Section 2 if you decide to add a pinned `requirements.txt` to the repository. Do not claim a package lockfile exists unless you commit one.

### LinkedIn draft — publish only after the notebook is verified and committed

**Unwrapping AI | Zero to Transformer — Step 03: Linear Regression**

How does a machine learning model learn a relationship between input features and a continuous outcome?

In today's hands-on milestone, I explored Linear Regression using the California Housing dataset.

The practical exercise covers:
- Understanding slope, intercept, and the regression equation.
- Building a simple model using one feature.
- Extending the model to multiple input features.
- Comparing predictions with a mean baseline.
- Evaluating performance using MSE, RMSE, MAE, and R².
- Visualizing actual vs predicted values and model residuals.

**Key learning:** Linear regression gives us an interpretable starting point for prediction, but evaluation on unseen data and careful interpretation of coefficients are essential.

This is Step 03 of my 12-step learning journey from machine learning fundamentals to Transformers.

GitHub: https://github.com/MeNitinsethi/Zero-to-transformer

**Unwrapping AI — Learn the concept. Build the model. Understand the why.**

#UnwrappingAI #MachineLearning #LinearRegression #Python #DataScience

*Edit the post to include a specific result or insight from your run. Publish only after the notebook is verified and saved on GitHub.*